# Met-3DNet-VI v0.3 — Functional Label Upgrade
## TumorAgDB1.0 Integration + Retrain

**What this notebook does in order:**
1. Audits all TumorAgDB1.0 xlsx files and extracts usable labels
2. Merges with existing IEDB-derived training splits  
3. Reports label distribution before/after
4. Retrains the model with real functional labels replacing pseudo-unknowns
5. Evaluates on the same held-out test set for fair comparison

**Expected improvement:** functional accuracy from 97.3% (majority-class artefact)
to genuine 3-class discrimination once 812+ human activating labels are added.

## Cell 1 — Setup

In [1]:
import os, sys, glob, shutil, subprocess, re
import warnings; warnings.filterwarnings("ignore")

KAGGLE   = os.path.exists("/kaggle/input")
WORK_DIR = "/kaggle/working" if KAGGLE else os.getcwd()
MODELS_DIR = os.path.join(WORK_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

subprocess.run(["pip","install","pyarrow","torch-geometric",
                "scikit-learn","openpyxl","--quiet"], check=True)

import torch, numpy as np, pandas as pd
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Locate Kaggle dataset roots ────────────────────────────
INPUT_ROOTS = []
if KAGGLE:
    for ds in os.listdir("/kaggle/input"):
        INPUT_ROOTS.append(os.path.join("/kaggle/input", ds))

def find_file(name, roots=None):
    """Find a file by name (or glob pattern) across all dataset roots."""
    search_roots = roots or INPUT_ROOTS or [WORK_DIR]
    for root in search_roots:
        for p in glob.glob(os.path.join(root, "**", name), recursive=True):
            if os.path.exists(p):
                return p
    return None

# Copy model checkpoint
for ds_root in INPUT_ROOTS:
    src = os.path.join(ds_root, "best_model.pt")
    if os.path.exists(src):
        dst = os.path.join(MODELS_DIR, "best_model.pt")
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        break

print(f"Setup complete ✓  |  {len(INPUT_ROOTS)} Kaggle datasets mounted")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.7 MB/s eta 0:00:00
Device: cuda
Setup complete ✓  |  1 Kaggle datasets mounted


## Cell 2 — Audit TumorAgDB1.0 Files
Reads every xlsx file, prints its sheets and columns, identifies peptide + HLA + label columns.
Run this first to understand the data before extraction.

In [2]:
from openpyxl import load_workbook

# All TumorAgDB1.0 file names (from the dataset description)
XLSX_NAMES = [
    "All the data on T cell activation experiments.xlsx",
    "NeoAntigen-PubData 2024-2025.xlsx",
    "Non-immunogenic Mutation Dataset.xlsx",
    "T-mTSA-negative-mouse.xlsx",
    "T-mTSA-postive-Homo sapiens-812.xlsx",
    "T-mTSA-postive-mouse.xlsx",
    "Tumor Protein Database.xlsx",
    "Validated Immunogenic Neoantigen Data.xlsx",
    "immunogenic Mutation Dataset.xlsx",
    "immunogenic Neo-peptide Dataset.xlsx",
    "mouse-MHC I-MB49-B16F10-P815-BBN963.xlsx",
    "mouse-MHC II-MB49-B16F10.xlsx",
]

# Keywords to look for in column names
PEP_KEYS  = ["peptide","sequence","epitope","neo","mutation","mutant","antigen"]
HLA_KEYS  = ["hla","mhc","allele","restriction","locus"]
LABEL_KEYS= ["immuno","response","positive","negative","label","class",
              "activation","ifn","elispot","validated","functional"]

def audit_xlsx(path):
    """Read xlsx and return sheet summary + column classifications."""
    wb = load_workbook(path, read_only=True, data_only=True)
    results = {}
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = list(ws.iter_rows(max_row=3, values_only=True))
        if not rows: continue
        header = [str(c).strip() if c else "" for c in rows[0]]
        sample  = rows[1] if len(rows) > 1 else []

        pep_cols   = [h for h in header if any(k in h.lower() for k in PEP_KEYS)]
        hla_cols   = [h for h in header if any(k in h.lower() for k in HLA_KEYS)]
        label_cols = [h for h in header if any(k in h.lower() for k in LABEL_KEYS)]

        results[sheet] = {
            "n_cols"     : len(header),
            "pep_cols"   : pep_cols[:5],
            "hla_cols"   : hla_cols[:5],
            "label_cols" : label_cols[:5],
            "header"     : header[:12],
            "sample"     : [str(v)[:40] if v else "" for v in (sample or [])[:8]],
        }
    wb.close()
    return results

print("Auditing TumorAgDB1.0 files...")
print("="*70)

AUDIT = {}
for fname in XLSX_NAMES:
    path = find_file(fname)
    if path is None:
        print(f"  NOT FOUND: {fname}")
        continue

    sz = os.path.getsize(path) / 1e6
    try:
        info = audit_xlsx(path)
        AUDIT[fname] = {"path": path, "size_mb": sz, "sheets": info}

        # Score usefulness
        has_pep   = any(s["pep_cols"]   for s in info.values())
        has_hla   = any(s["hla_cols"]   for s in info.values())
        has_label = any(s["label_cols"] for s in info.values())
        usability = sum([has_pep, has_hla, has_label])
        flag = "HIGH" if usability == 3 else "MED" if usability >= 2 else "LOW"

        print(f"\n[{flag}] {fname}  ({sz:.1f} MB)")
        for sheet, s in info.items():
            print(f"  Sheet: {sheet}")
            print(f"    Peptide cols : {s['pep_cols'] or 'none found'}")
            print(f"    HLA cols     : {s['hla_cols'] or 'none found'}")
            print(f"    Label cols   : {s['label_cols'] or 'none found'}")
            print(f"    All cols     : {s['header']}")
    except Exception as e:
        print(f"  ERROR reading {fname}: {e}")

print("\nAudit complete ✓")

Auditing TumorAgDB1.0 files...

[HIGH] All the data on T cell activation experiments.xlsx  (13.8 MB)
  Sheet: Sheet1
    Peptide cols : ['Antigen_Name', 'Epitope_Antigen_IRI', 'Epitope_Organism', 'Length_of_Peptide', 'Peptide']
    HLA cols     : ['MHC_Allele', 'MHC_Type']
    Label cols   : ['immunogenicity']
    All cols     : ['Antigen_Name', 'Assay_Group', 'Assay_Units', 'Epitope_Antigen_IRI', 'Epitope_Organism', 'Length_of_Peptide', 'MHC_Allele', 'MHC_Type', 'Peptide', 'Protein_IRI', 'Qualitative_Measure', 'Reference']

[LOW] NeoAntigen-PubData 2024-2025.xlsx  (0.8 MB)

[HIGH] Non-immunogenic Mutation Dataset.xlsx  (5.7 MB)
  Sheet: Sheet1
    Peptide cols : ['aa_mutant', 'mutant_seq', 'mutation_type', 'nb_same_mutation_Intogen', 'nb_mutations_in_gene_Intogen']
    HLA cols     : ['COUNT_MUT_RANK_CI_MIXMHC', 'COUNT_MUT_RANK_CI_netMHCpan', 'MIN_MUT_RANK_CI_MIXMHC', 'WT_BEST_RANK_CI_MIXMHC']
    Label cols   : ['response_type', 'immunogenicity']
    All cols     : ['dataset', 'respo

## Cell 3 — Extract Functional Labels from Each File
After reviewing the audit output above, edit the column mappings below if needed.
The defaults are best guesses based on TumorAgDB1.0 standard schema.

In [3]:
VALID_AA   = set("ACDEFGHIKLMNPQRSTVWY")
HLA_PATTERN = re.compile(r"HLA-[ABC]\*\d{2}:\d{2}")

def valid_pep(seq, mn=8, mx=14):
    if not isinstance(seq, str): return False
    s = seq.strip().upper()
    return mn <= len(s) <= mx and all(c in VALID_AA for c in s)

def norm_hla(s):
    """Normalise HLA string to HLA-X*##:## format."""
    if not isinstance(s, str): return None
    s = s.strip()
    m = HLA_PATTERN.search(s)
    if m: return m.group()
    # Try to repair common formats: A*02:01, HLA-A02:01, A0201
    s = re.sub(r"^(A|B|C)(\*?)(\d{2})(\:?)(\d{2})$",
               lambda m: f"HLA-{m.group(1)}*{m.group(3)}:{m.group(5)}", s)
    m = HLA_PATTERN.search(s)
    return m.group() if m else None

def read_sheet_flexible(path, sheet_name=None, max_rows=None):
    """Read xlsx sheet into DataFrame, flexible column detection."""
    kwargs = dict(engine="openpyxl")
    if sheet_name: kwargs["sheet_name"] = sheet_name
    if max_rows:   kwargs["nrows"] = max_rows
    try:
        return pd.read_excel(path, **kwargs)
    except Exception:
        return pd.DataFrame()

def find_col(df, keywords):
    """Find first column whose name contains any keyword."""
    for col in df.columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

def extract_records(path, functional_class, source_tag, max_rows=None):
    """
    Extract (peptide, hla_allele, functional_class) triples from an xlsx file.
    functional_class: 2=activating, 1=suppressive, 0=non-immunogenic
    """
    records = []
    try:
        xl = pd.ExcelFile(path, engine="openpyxl")
    except Exception as e:
        print(f"  Cannot open {os.path.basename(path)}: {e}")
        return pd.DataFrame()

    for sheet in xl.sheet_names:
        try:
            df = xl.parse(sheet, nrows=max_rows)
        except Exception:
            continue

        if len(df) == 0 or len(df.columns) == 0:
            continue

        pep_col   = find_col(df, ["peptide","sequence","epitope","neo_peptide",
                                   "mutant_peptide","neopeptide","neo-peptide"])
        hla_col   = find_col(df, ["hla","mhc","allele","restriction"])

        if pep_col is None:
            # Try positional: first text column with 8-14 char strings
            for col in df.columns:
                sample = df[col].dropna().head(20).astype(str)
                if sample.apply(lambda x: valid_pep(x.strip().upper())).mean() > 0.5:
                    pep_col = col
                    break

        if pep_col is None:
            continue

        for _, row in df.iterrows():
            pep = str(row.get(pep_col, "")).strip().upper()
            if not valid_pep(pep):
                continue

            hla = None
            if hla_col:
                hla = norm_hla(str(row.get(hla_col, "")))
            if hla is None:
                # Scan all columns for an HLA string
                for col in df.columns:
                    candidate = norm_hla(str(row.get(col, "")))
                    if candidate:
                        hla = candidate
                        break
            if hla is None:
                hla = "HLA-A*02:01"   # default for unspecified

            records.append({
                "peptide"          : pep,
                "hla_allele"       : hla,
                "functional_class" : functional_class,
                "immunogenicity"   : 1 if functional_class in [1, 2] else 0,
                "source"           : source_tag,
                "peptide_len"      : len(pep),
            })

    return pd.DataFrame(records)

# ── Extract from each HIGH/MED priority file ─────────────────

print("Extracting records from TumorAgDB1.0...")
dfs = []

# Activating (class=2): human positive records
for fname, tag in [
    ("T-mTSA-postive-Homo sapiens-812.xlsx",   "tumoragdb_human_pos"),
    ("Validated Immunogenic Neoantigen Data.xlsx","tumoragdb_validated"),
    ("immunogenic Neo-peptide Dataset.xlsx",    "tumoragdb_neo_pep"),
    ("NeoAntigen-PubData 2024-2025.xlsx",       "tumoragdb_2024_2025"),
]:
    path = find_file(fname)
    if path:
        df = extract_records(path, functional_class=2, source_tag=tag)
        if len(df):
            dfs.append(df)
            print(f"  ✓ {fname}: {len(df):,} activating records")
    else:
        print(f"  ✗ NOT FOUND: {fname}")

# Non-immunogenic (class=0): confirmed negatives
for fname, tag in [
    ("Non-immunogenic Mutation Dataset.xlsx", "tumoragdb_nonimmuno"),
]:
    path = find_file(fname)
    if path:
        df = extract_records(path, functional_class=0, source_tag=tag)
        if len(df):
            dfs.append(df)
            print(f"  ✓ {fname}: {len(df):,} non-immunogenic records")

# T cell activation experiments — check label column
fname = "All the data on T cell activation experiments.xlsx"
path  = find_file(fname)
if path:
    xl = pd.ExcelFile(path, engine="openpyxl")
    for sheet in xl.sheet_names:
        df_raw = xl.parse(sheet, nrows=5000)
        label_col = find_col(df_raw, ["positive","negative","result",
                                       "response","immuno","outcome"])
        pep_col   = find_col(df_raw, ["peptide","sequence","epitope"])
        if pep_col and label_col:
            act_mask = df_raw[label_col].astype(str).str.lower().str.contains(
                "positive|yes|true|1|activat|ifn", na=False)
            sup_mask = df_raw[label_col].astype(str).str.lower().str.contains(
                "suppress|il-10|treg|regulatory", na=False)
            neg_mask = df_raw[label_col].astype(str).str.lower().str.contains(
                "negative|no|false|0|non.immuno", na=False)
            hla_col  = find_col(df_raw, ["hla","mhc","allele"])
            for fc, mask, tag in [(2, act_mask, "tcell_activating"),
                                   (1, sup_mask, "tcell_suppressive"),
                                   (0, neg_mask, "tcell_negative")]:
                sub = df_raw[mask].copy()
                if len(sub) == 0: continue
                recs = []
                for _, row in sub.iterrows():
                    pep = str(row[pep_col]).strip().upper()
                    if not valid_pep(pep): continue
                    hla = norm_hla(str(row[hla_col])) if hla_col else "HLA-A*02:01"
                    recs.append({"peptide":pep,"hla_allele":hla or "HLA-A*02:01",
                                  "functional_class":fc,"immunogenicity":1 if fc>0 else 0,
                                  "source":tag,"peptide_len":len(pep)})
                if recs:
                    dfs.append(pd.DataFrame(recs))
                    print(f"  ✓ {fname} [{sheet}] class={fc}: {len(recs):,} records")

# ── Combine all new records ──────────────────────────────────
if not dfs:
    print("\nWARNING: No records extracted. Check that xlsx files are in the Kaggle dataset.")
    new_data = pd.DataFrame(columns=["peptide","hla_allele","functional_class",
                                      "immunogenicity","source","peptide_len"])
else:
    new_data = pd.concat(dfs, ignore_index=True)
    new_data = new_data.drop_duplicates(subset=["peptide","hla_allele"])
    print(f"\nTotal new records extracted: {len(new_data):,}")
    print(f"  Activating  (class=2): {(new_data['functional_class']==2).sum():,}")
    print(f"  Suppressive (class=1): {(new_data['functional_class']==1).sum():,}")
    print(f"  Non-immunogenic (0)  : {(new_data['functional_class']==0).sum():,}")
    print(f"  Unique HLA alleles   : {new_data['hla_allele'].nunique()}")
    print(f"  Sources: {new_data['source'].value_counts().to_dict()}")

Extracting records from TumorAgDB1.0...



## Cell 4 — Encode New Records + Merge with Existing Splits
Applies the same AAindex encoding used during original training.
Matches new records to existing splits or creates a new supplementary training set.

In [ ]:
# ── AAindex encoding (identical to training pipeline) ────────
AA_PROPS = {
    "A":[ 1.8, 0.0, 88.6,0.360, 8.1],"R":[-4.5, 1.0,173.4,0.530,10.5],
    "N":[-3.5, 0.0,114.1,0.460,11.6],"D":[-3.5,-1.0,111.1,0.510,13.0],
    "C":[ 2.5, 0.0,108.5,0.350, 5.5],"Q":[-3.5, 0.0,143.8,0.490,10.5],
    "E":[-3.5,-1.0,138.4,0.500,12.3],"G":[-0.4, 0.0, 60.1,0.540, 5.7],
    "H":[-3.2, 0.5,153.2,0.320, 8.4],"I":[ 4.5, 0.0,166.7,0.460, 5.2],
    "L":[ 3.8, 0.0,166.7,0.450, 4.9],"K":[-3.9, 1.0,168.6,0.470,10.1],
    "M":[ 1.9, 0.0,162.9,0.360, 5.4],"F":[ 2.8, 0.0,189.9,0.310, 5.2],
    "P":[-1.6, 0.0,112.7,0.000, 8.0],"S":[-0.8, 0.0, 89.0,0.510, 9.2],
    "T":[-0.7, 0.0,116.1,0.440, 8.6],"W":[-0.9, 0.0,227.8,0.310, 5.4],
    "Y":[-1.3, 0.0,193.6,0.420, 6.2],"V":[ 4.2, 0.0,140.0,0.390, 5.9],
}

HLA_PSEUDO = {
    "HLA-A*02:01": "YYAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*01:01": "YSAMHQENMAYTDANTLYIIYRDAQTFRVD",
    "HLA-A*03:01": "YFAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*11:01": "YFAMYQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-A*24:02": "YYAMFQENMAHTDANTLYIIYRDAQTFRVD",
    "HLA-B*07:02": "HSMRYNSTAVYLENMAATYIIIRDAQTFRV",
    "HLA-B*35:01": "HSLRYHSTAVYLENMAATYIIIRDAQTFRV",
    "HLA-B*57:01": "HSLRYHSTAVYLENMAHSDAIIIRDAQTFR",
}
HLA_PSEUDO = {k: (v+"A"*34)[:34] for k, v in HLA_PSEUDO.items()}
DEFAULT_HLA = "HLA-A*02:01"

def encode_seq(seq, max_len):
    arr = np.zeros((max_len, 5), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        if aa in AA_PROPS: arr[i] = AA_PROPS[aa]
    return arr.flatten()

def compute_feats(pep):
    valid = [aa for aa in pep if aa in AA_PROPS]
    if not valid: return [0.,0.,0.,float(len(pep))]
    return [float(np.mean([AA_PROPS[a][0] for a in valid])),
            float(sum(AA_PROPS[a][1] for a in valid)),
            float(np.mean([AA_PROPS[a][2] for a in valid])),
            float(len(pep))]

def encode_row(pep, hla_allele):
    hla_seq    = HLA_PSEUDO.get(hla_allele, HLA_PSEUDO[DEFAULT_HLA])
    peptide_enc = encode_seq(pep, 14)
    hla_enc     = encode_seq(hla_seq, 34)
    feats       = compute_feats(pep)
    return peptide_enc, hla_enc, feats

# ── Encode new_data ──────────────────────────────────────────
if len(new_data) > 0:
    print(f"Encoding {len(new_data):,} new records...")
    pep_encs, hla_encs, feat_hydros, feat_charges, feat_volumes, feat_lens =         [], [], [], [], [], []

    for _, row in new_data.iterrows():
        pe, he, feats = encode_row(row["peptide"], row["hla_allele"])
        pep_encs.append(pe); hla_encs.append(he)
        feat_hydros.append(feats[0]); feat_charges.append(feats[1])
        feat_volumes.append(feats[2]); feat_lens.append(feats[3])

    new_data = new_data.copy()
    new_data["peptide_enc"] = pep_encs
    new_data["hla_enc"]     = hla_encs
    new_data["feat_hydro"]  = feat_hydros
    new_data["feat_charge"] = feat_charges
    new_data["feat_volume"] = feat_volumes
    new_data["feat_len"]    = feat_lens
    new_data["data_source"] = new_data["source"]
    new_data["hla_seq"]     = new_data["hla_allele"].map(
        lambda a: "".join(chr(int(x)) if x.isdigit() else x
                          for x in []) or HLA_PSEUDO.get(a, HLA_PSEUDO[DEFAULT_HLA]))
    print("Encoding complete ✓")

# ── Load existing upgraded training split ────────────────────
train_path = None
for fname in ["split_train_upgraded.parquet","split_train.parquet"]:
    p = find_file(fname)
    if p: train_path = p; break

if train_path is None:
    raise FileNotFoundError("split_train*.parquet not found. Upload the upgraded parquet files.")

train_orig = pd.read_parquet(train_path)
val_orig   = pd.read_parquet(find_file("split_val_upgraded.parquet")
                              or find_file("split_val.parquet"))
test_orig  = pd.read_parquet(find_file("split_test_upgraded.parquet")
                              or find_file("split_test.parquet"))

print(f"\nOriginal training split: {len(train_orig):,} rows")
print(f"  Activating  (2): {(train_orig['functional_class']==2).sum()}")
print(f"  Suppressive (1): {(train_orig['functional_class']==1).sum()}")
print(f"  Unknown     (0): {(train_orig['functional_class']==0).sum()}")

# ── Upgrade functional labels in existing data using new records ─
# Strategy: for existing rows with functional_class=0 (unknown),
# check if the same peptide appears in new_data with a real label
print("\nUpgrading labels in existing training data...")
new_lookup = {}
if len(new_data) > 0:
    for _, row in new_data.iterrows():
        key = (row["peptide"], row["hla_allele"])
        # Priority: activating > suppressive > non-immunogenic
        existing = new_lookup.get(key, -1)
        new_lookup[key] = max(existing, row["functional_class"])

upgraded_count = 0
train_new = train_orig.copy()
for idx, row in train_new.iterrows():
    if row["functional_class"] == 0:   # only upgrade unknowns
        key = (row["peptide"], row["hla_allele"])
        if key in new_lookup and new_lookup[key] > 0:
            train_new.at[idx, "functional_class"] = new_lookup[key]
            upgraded_count += 1

print(f"  Upgraded {upgraded_count:,} unknown→labeled in existing training data")

# ── Add truly new peptides to training set ───────────────────
# Only add new_data records NOT already in train/val/test
existing_keys = set()
for df in [train_orig, val_orig, test_orig]:
    for _, row in df.iterrows():
        existing_keys.add((row["peptide"], row["hla_allele"]))

genuinely_new = new_data[new_data.apply(
    lambda r: (r["peptide"], r["hla_allele"]) not in existing_keys, axis=1
)].copy() if len(new_data) > 0 else pd.DataFrame()

print(f"  Genuinely new peptides to add: {len(genuinely_new):,}")

# Align columns
if len(genuinely_new) > 0:
    for col in train_orig.columns:
        if col not in genuinely_new.columns:
            genuinely_new[col] = None
    genuinely_new = genuinely_new[train_orig.columns]
    train_combined = pd.concat([train_new, genuinely_new], ignore_index=True)
else:
    train_combined = train_new

print(f"\nCombined training set: {len(train_combined):,} rows")
print(f"  Activating  (2): {(train_combined['functional_class']==2).sum():,}")
print(f"  Suppressive (1): {(train_combined['functional_class']==1).sum():,}")
print(f"  Unknown     (0): {(train_combined['functional_class']==0).sum():,}")
print(f"  Negative immuno: {(train_combined['immunogenicity']==0).sum():,}")

# Save upgraded splits
out_train = os.path.join(WORK_DIR, "split_train_v3.parquet")
train_combined.to_parquet(out_train, index=False)
print(f"\nSaved: {out_train}")

## Cell 5 — Model Architecture + Graph Construction

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader

MAX_PEP=14; MAX_HLA=34; N_FEAT=5; N_NODES=49

def build_node_features(pep_enc, hla_enc):
    pep  = np.array(pep_enc,dtype=np.float32).reshape(MAX_PEP,N_FEAT)
    hla  = np.array(hla_enc,dtype=np.float32).reshape(MAX_HLA,N_FEAT)
    phys = np.zeros((N_NODES,N_FEAT),dtype=np.float32)
    phys[:MAX_PEP]=pep; phys[MAX_PEP:MAX_PEP+MAX_HLA]=hla
    nt   = np.zeros((N_NODES,3),dtype=np.float32)
    nt[:MAX_PEP,0]=1; nt[MAX_PEP:MAX_PEP+MAX_HLA,1]=1; nt[-1,2]=1
    return torch.tensor(np.concatenate([phys,nt],axis=1))

def build_edges(plen):
    v=N_NODES-1; src,dst,et=[],[],[]
    for i in range(plen-1): src+=[i,i+1];dst+=[i+1,i];et+=[0,0]
    for i in range(MAX_PEP,MAX_PEP+MAX_HLA-1): src+=[i,i+1];dst+=[i+1,i];et+=[0,0]
    for i in range(N_NODES-1): src+=[i,v];dst+=[v,i];et+=[2,2]
    ei = torch.tensor([src,dst],dtype=torch.long)
    ea = torch.zeros(len(src),3)
    for i,t in enumerate(et): ea[i,t]=1.0
    return ei,ea

def row_to_graph(row):
    x  = build_node_features(row["peptide_enc"],row["hla_enc"])
    ei,ea = build_edges(int(row["peptide_len"]))
    gf = torch.tensor([float(row["feat_hydro"]),float(row["feat_charge"]),
                        float(row["feat_volume"]),float(row["feat_len"])],
                       dtype=torch.float).unsqueeze(0)
    return Data(x=x,edge_index=ei,edge_attr=ea,graph_feat=gf,
                y_immuno=torch.tensor([row["immunogenicity"]],dtype=torch.float),
                y_func=torch.tensor([row["functional_class"]],dtype=torch.long),
                peptide=row.get("peptide",""), hla_allele=row.get("hla_allele",""),
                num_nodes=N_NODES)

class NeoantigenGraphDataset(Dataset):
    def __init__(self,df_or_path):
        super().__init__()
        self.df = (pd.read_parquet(df_or_path) if isinstance(df_or_path,str)
                   else df_or_path).reset_index(drop=True)
        print(f"  Loaded {len(self.df):,} samples")
    def len(self): return len(self.df)
    def get(self,idx): return row_to_graph(self.df.iloc[idx])

class FiLMLayer(nn.Module):
    def __init__(self,h,c=4):
        super().__init__()
        self.g=nn.Sequential(nn.Linear(c,h),nn.ReLU(),nn.Linear(h,h))
        self.b=nn.Sequential(nn.Linear(c,h),nn.ReLU(),nn.Linear(h,h))
        self.n=nn.LayerNorm(h)
    def forward(self,x,ctx,batch):
        c=ctx[batch]; return self.n(self.g(c)*x+self.b(c))

class Met3DNetVI(nn.Module):
    def __init__(self,hidden_dim=128,dropout=0.1):
        super().__init__()
        self.node_proj=nn.Sequential(nn.Linear(8,hidden_dim),nn.LayerNorm(hidden_dim),
                                      nn.GELU(),nn.Dropout(dropout))
        for i in range(1,4):
            setattr(self,f"gnn{i}",GATConv(hidden_dim,hidden_dim,heads=4,concat=False,dropout=dropout))
            setattr(self,f"norm{i}",nn.LayerNorm(hidden_dim))
        self.film=FiLMLayer(hidden_dim)
        def _h(o): return nn.Sequential(nn.Linear(hidden_dim,hidden_dim//2),
                                         nn.LayerNorm(hidden_dim//2),nn.GELU(),
                                         nn.Dropout(dropout),nn.Linear(hidden_dim//2,o))
        self.head_immuno=_h(1); self.head_func=_h(3); self.head_score=_h(1)
    def forward(self,data):
        x,ei,batch=data.x,data.edge_index,data.batch
        h=self.node_proj(x)
        for i in range(1,4):
            h=getattr(self,f"norm{i}")(h+F.relu(getattr(self,f"gnn{i}")(h,ei)))
        ctx=data.graph_feat.squeeze(1)
        h=self.film(h,ctx,batch)
        B=batch.max().item()+1
        vi=torch.tensor([b*N_NODES+N_NODES-1 for b in range(B)],device=x.device)
        g=h[vi]
        return {"logit_immuno":self.head_immuno(g),"logit_func":self.head_func(g),
                "score_activate":self.head_score(g),"graph_emb":g}

def build_model(cfg): return Met3DNetVI(hidden_dim=cfg.get("hidden_dim",128),
                                         dropout=cfg.get("dropout",0.1))
print("Model architecture defined ✓")

## Cell 6 — Training Config v0.3
Key changes: `lambda2` restored to 0.5 now that we have real functional labels. `func_weights` rebalanced for new class distribution.

In [ ]:
# Compute class weights dynamically from new training distribution
n_unknown = int((train_combined["functional_class"]==0).sum())
n_suppress= int((train_combined["functional_class"]==1).sum())
n_activat = int((train_combined["functional_class"]==2).sum())
n_total_fc= n_unknown + n_suppress + n_activat
print(f"Functional class distribution in combined training set:")
print(f"  Unknown     (0): {n_unknown:>6,}  ({n_unknown/n_total_fc*100:.1f}%)")
print(f"  Suppressive (1): {n_suppress:>6,}  ({n_suppress/n_total_fc*100:.1f}%)")
print(f"  Activating  (2): {n_activat:>6,}  ({n_activat/n_total_fc*100:.1f}%)")

# Inverse frequency weights — cap at 20x to avoid instability
def inv_freq_weight(n, total, cap=20.0):
    return min(total / (3 * max(n, 1)), cap)

w0 = inv_freq_weight(n_unknown,  n_total_fc)
w1 = inv_freq_weight(n_suppress, n_total_fc)
w2 = inv_freq_weight(n_activat,  n_total_fc)
print(f"\nAuto-computed class weights: neutral={w0:.2f} suppressive={w1:.2f} activating={w2:.2f}")

CONFIG = {
    "hidden_dim"   : 128,
    "dropout"      : 0.1,
    "lr"           : 5e-4,          # slightly lower for fine-tuning
    "weight_decay" : 1e-4,
    "batch_size"   : 64 if torch.cuda.is_available() else 16,
    "epochs"       : 80,
    "patience"     : 12,
    "lambda1"      : 1.0,           # immunogenicity BCE
    "lambda2"      : 0.5,           # functional CE — restored now that labels exist
    "lambda3"      : 0.3,           # ranking loss
    "pos_weight"   : 3.0,
    "func_weights" : [round(w0,2), round(w1,2), round(w2,2)],
    "seed"         : 42,
    "version"      : "v0.3.0",
    "note"         : "TumorAgDB1.0 functional labels integrated",
}
print(f"\nCONFIG: {CONFIG}")

## Cell 7 — Train v0.3
Fine-tunes from best_model.pt checkpoint (warm start) OR trains from scratch if not found.

In [ ]:
import time, json
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

torch.manual_seed(CONFIG["seed"])

# ── Datasets ────────────────────────────────────────────────
print("Building graph datasets...")
train_ds = NeoantigenGraphDataset(train_combined)
val_ds   = NeoantigenGraphDataset(val_orig)
test_ds  = NeoantigenGraphDataset(test_orig)

tl  = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
vl  = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
tsl = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

# ── Model: warm start from v0.2 checkpoint ─────────────────
model = build_model(CONFIG).to(device)
ckpt_path = os.path.join(MODELS_DIR, "best_model.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Warm start from v0.2 checkpoint (val AUC={ckpt.get('val_auc',0):.4f})")
else:
    print("Training from scratch (no checkpoint found)")

n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_p:,}")

optimiser = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                               weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=CONFIG["epochs"], eta_min=1e-6)

bce = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([CONFIG["pos_weight"]]).to(device))
fw  = torch.tensor(CONFIG["func_weights"], dtype=torch.float).to(device)
ce  = nn.CrossEntropyLoss(weight=fw, ignore_index=-1)

def compute_loss(out, batch):
    logit = out["logit_immuno"].squeeze(-1)
    y_im  = batch.y_immuno.to(device).squeeze(-1).float()
    y_fn  = batch.y_func.to(device).squeeze(-1).long()
    sc    = out["score_activate"].squeeze(-1)
    l1 = bce(logit, y_im)
    mask = y_fn >= 0
    l2 = ce(out["logit_func"][mask], y_fn[mask]) if mask.sum()>0          else torch.tensor(0., device=device)
    pos=y_im==1; neg=y_im==0
    l3 = torch.clamp(1.-sc[pos].unsqueeze(1)+sc[neg].unsqueeze(0),min=0).mean()          if pos.sum()>0 and neg.sum()>0 else torch.tensor(0.,device=device)
    return CONFIG["lambda1"]*l1 + CONFIG["lambda2"]*l2 + CONFIG["lambda3"]*l3

def evaluate(loader):
    model.eval()
    pa,ya,fp,ft=[],[],[],[]
    tl=0.
    with torch.no_grad():
        for b in loader:
            b=b.to(device); out=model(b)
            tl+=compute_loss(out,b).item()
            pa.extend(torch.sigmoid(out["logit_immuno"].squeeze(-1)).cpu().numpy())
            fp.extend(out["logit_func"].argmax(1).cpu().numpy())
            ya.extend(b.y_immuno.squeeze(-1).cpu().numpy())
            ft.extend(b.y_func.squeeze(-1).cpu().numpy())
    pa=np.array(pa);ya=np.array(ya);fp=np.array(fp);ft=np.array(ft)
    pred=(pa>=0.42).astype(int)
    try: auc=roc_auc_score(ya,pa)
    except: auc=0.
    f1=f1_score(ya,pred,zero_division=0)
    acc=accuracy_score(ya,pred)
    lab=ft>=0
    facc=accuracy_score(ft[lab],fp[lab]) if lab.sum()>0 else 0.
    # Per-class functional accuracy
    fc_detail={}
    for fc,nm in [(0,"unknown"),(1,"suppressive"),(2,"activating")]:
        m=ft==fc
        fc_detail[nm]=round(accuracy_score(ft[m],fp[m]),4) if m.sum()>0 else None
    return {"loss":round(tl/len(loader),4),"auc":round(float(auc),4),
            "f1":round(float(f1),4),"acc":round(float(acc),4),
            "func_acc":round(float(facc),4),"fc_detail":fc_detail}

# ── Training ─────────────────────────────────────────────────
best_auc=0.; patience_count=0; history=[]; save_path=os.path.join(MODELS_DIR,"best_model_v3.pt")
print(f"\n{'Epoch':>5}  {'TrLoss':>8}  {'ValAUC':>7}  {'F1':>7}  {'FuncAcc':>8}  {'ActAcc':>7}")
print("-"*60)

for epoch in range(1, CONFIG["epochs"]+1):
    t0=time.time(); model.train(); tr=0.
    for b in tl:
        b=b.to(device); optimiser.zero_grad()
        loss=compute_loss(model(b),b); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimiser.step(); tr+=loss.item()
    tr/=len(tl); scheduler.step()
    vm=evaluate(vl)
    act_acc=vm["fc_detail"].get("activating")
    act_str=f"{act_acc:.4f}" if act_acc is not None else "  n/a "
    print(f"{epoch:>5}  {tr:>8.4f}  {vm['auc']:>7.4f}  {vm['f1']:>7.4f}  "
          f"{vm['func_acc']:>8.4f}  {act_str}  ({time.time()-t0:.1f}s)")
    history.append({"epoch":epoch,"train_loss":round(tr,4),
                    **{f"val_{k}":v for k,v in vm.items() if k!="fc_detail"}})
    if vm["auc"]>best_auc:
        best_auc=vm["auc"]; patience_count=0
        torch.save({"state_dict":model.state_dict(),"val_auc":best_auc,
                    "config":CONFIG},save_path)
        print(f"  ✓ Best AUC={best_auc:.4f} — saved")
    else:
        patience_count+=1
        if patience_count>=CONFIG["patience"]:
            print(f"\nEarly stopping at epoch {epoch}"); break

# ── Test evaluation ───────────────────────────────────────────
print("\nLoading best v0.3 checkpoint...")
ckpt=torch.load(save_path,map_location=device,weights_only=False)
model.load_state_dict(ckpt["state_dict"])
test_metrics=evaluate(tsl)
print("\n"+"="*55)
print("TEST RESULTS — v0.3 vs v0.2 comparison")
print("="*55)
v2={"auc":0.7562,"f1":0.5203,"acc":0.665,"func_acc":0.9733}
for k in ["auc","f1","acc","func_acc"]:
    v=test_metrics[k]; old=v2[k]
    delta=v-old; arrow="↑" if delta>0 else "↓"
    print(f"  {k:<12}: {v:.4f}  ({arrow}{abs(delta):.4f} vs v0.2 {old:.4f})")
print("\nPer-class functional accuracy:")
for cls,acc in test_metrics.get("fc_detail",{}).items():
    print(f"  {cls:<15}: {acc:.4f}" if acc else f"  {cls:<15}: N/A (no samples)")

hist_data={"history":history,"test":test_metrics,"config":CONFIG}
with open(os.path.join(MODELS_DIR,"training_history_v3.json"),"w") as f:
    json.dump(hist_data,f,indent=2)
print(f"\nHistory saved: training_history_v3.json")

## Cell 8 — Training Curves + v0.2 vs v0.3 Comparison

In [ ]:
import matplotlib.pyplot as plt

history = hist_data["history"]
epochs  = [h["epoch"]     for h in history]
tr_loss = [h["train_loss"] for h in history]
val_auc = [h["val_auc"]    for h in history]
val_f1  = [h["val_f1"]     for h in history]
val_fa  = [h.get("val_func_acc",0) for h in history]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(epochs, tr_loss, color="#185FA5", lw=2, label="v0.3")
axes[0].axhline(1.3356, ls="--", color="gray", alpha=0.5, label="v0.2 test loss")
axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_auc, color="#1D9E75", lw=2, label="v0.3 val AUC")
axes[1].axhline(0.7562, ls="--", color="gray", alpha=0.5, label="v0.2 test AUC=0.756")
axes[1].set_ylim(0.5, 1.0); axes[1].set_title("Validation AUC")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, val_f1, color="#7F77DD", lw=2, label="v0.3 F1")
axes[2].axhline(0.5203, ls="--", color="gray", alpha=0.5, label="v0.2 F1=0.520")
axes[2].set_ylim(0, 1); axes[2].set_title("F1 (immunogenicity)")
axes[2].legend(); axes[2].grid(alpha=0.3)

axes[3].plot(epochs, val_fa, color="#D85A30", lw=2, label="v0.3 func acc")
axes[3].axhline(0.9733, ls="--", color="gray", alpha=0.5, label="v0.2=0.973 (artefact)")
axes[3].axhline(0.333, ls=":", color="silver", alpha=0.5, label="random baseline")
axes[3].set_ylim(0, 1.05); axes[3].set_title("Functional class accuracy")
axes[3].legend(fontsize=7); axes[3].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("Epoch")
plt.suptitle(f"Met-3DNet-VI v0.3 — TumorAgDB1.0 Integration",
             fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(WORK_DIR,"training_curves_v3.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves_v3.png")

# ── Final summary ────────────────────────────────────────────
print("\n"+"="*55)
print("LABEL UPGRADE SUMMARY")
print("="*55)
print(f"  v0.2 training data  : {len(train_orig):,} samples")
print(f"  v0.3 training data  : {len(train_combined):,} samples")
delta = len(train_combined) - len(train_orig)
print(f"  New records added   : {delta:+,}")
if len(new_data) > 0:
    print(f"  New activating      : {(new_data['functional_class']==2).sum():,}")
    print(f"  New suppressive     : {(new_data['functional_class']==1).sum():,}")
    print(f"  New non-immunogenic : {(new_data['functional_class']==0).sum():,}")
print(f"\n  v0.2 unknown labels : {(train_orig['functional_class']==0).sum():,} "
      f"({(train_orig['functional_class']==0).mean()*100:.1f}%)")
print(f"  v0.3 unknown labels : {(train_combined['functional_class']==0).sum():,} "
      f"({(train_combined['functional_class']==0).mean()*100:.1f}%)")
print(f"\n  Download: best_model_v3.pt + training_history_v3.json")